In [78]:
%matplotlib widget

from pathlib import Path
import math
import re

import pandas as pd
import tifffile
import matplotlib.pyplot as plt

In [ ]:
# -----------------------------
# Paths
# -----------------------------

BASE_DIR = Path.cwd().parents[1]

RAW_DIR = (
    BASE_DIR
    / "FIB_SEM_optimization"
    / "images"
    / "raw"
)

METADATA_FILE = (
    BASE_DIR
    / "FIB_SEM_optimization"
    / "FIB_SEM_Opt_datasets.csv"
)

COMPARISON_DIR = (
    BASE_DIR
    / "FIB_SEM_optimization"
    / "comparison_figures"
)

COMPARISON_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# -----------------------------
# Load metadata once
# -----------------------------

metadata = pd.read_csv(
    METADATA_FILE
)


# -----------------------------
# Extract image ID
# -----------------------------

def get_image_id(path):

    match = re.search(
        r"_LFP_(\d+)(?:_\d+)?$",
        path.stem
    )

    if match:
        return int(match.group(1))

    return None



# -----------------------------
# Plot function
# -----------------------------

def plot_fibsem_images(
    metadata_filter,
    title="FIB-SEM Images",
    filename="comparison",
    cols=4,
    suction_voltage=None,
    signal_out=None,
    Mode=None,    
):

    """
    Plot and save FIB-SEM images based on metadata filters.

    Example:

    plot_fibsem_images(
        (metadata["parameter_set"] == 2) &
        (metadata["current_nA"] == 0.1),
        title="Parameter Set 2 | Current 0.1 nA",
        filename="ParameterSet_2_Current_0.1nA"
    )
    """


    # Select metadata
    selected = metadata[
        metadata_filter
    ].sort_values(
        "image_id"
    )


    if len(selected) == 0:
        print("No images found")
        return


    wanted_ids = set(
        selected["image_id"]
    )


    # Find matching images
    image_paths = []

    for image_file in RAW_DIR.glob("*.tif"):

        image_id = get_image_id(
            image_file
        )

        if image_id in wanted_ids:
            image_paths.append(
                image_file
            )


    image_paths = sorted(
        image_paths,
        key=get_image_id
    )


    n_images = len(image_paths)


    if n_images == 0:
        print("No TIFF files found")
        return



    # Figure layout
    rows = math.ceil(
        n_images / cols
    )

    plt.close("all")
    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(30, 8 * rows),
        dpi=200
    )


    if n_images == 1:
        axes = [axes]

    else:
        axes = axes.flatten()



    # Plot images
    for ax, image_path in zip(
        axes,
        image_paths
    ):

        image = tifffile.imread(
            image_path
        )


        ax.imshow(
            image,
            cmap="gray",
            interpolation="nearest",
            aspect="equal"
        )


        image_id = get_image_id(
            image_path
        )


        meta_row = selected[
            selected["image_id"] == image_id
        ].iloc[0]


        ax.set_title(
            image_path.stem,
            fontsize=16
        )


        info = (
            f"ID: {image_id}\n"
            + (
                f"Suction Tube Voltage: {meta_row.get('suction_V', 'N/A')}\n"
                if suction_voltage is not None
                else ""
            )
            + (
                f"Mode: {meta_row.get('mode', 'N/A')}\n"
                if Mode is not None
                else ""
            )
            + f"Detector: {meta_row.get('detector', 'N/A')}\n"
            + (
                f"Signal Out: {meta_row.get('signal_out', 'N/A')}\n"
                if signal_out is not None
                else ""
            )
            + f"Current: {meta_row.get('current_nA', 'N/A')} nA\n"
            + f"Voltage: {meta_row.get('voltage_kV', 'N/A')} kV\n"
            + f"Magnification: {meta_row.get('magnification', 'N/A')}\n"
            + (
                f"Notes: {meta_row['notes']}"
                if pd.notna(meta_row["notes"])
                and str(meta_row["notes"]).strip().upper() != "NA"
                else ""
            )
        )


        ax.text(
            0.5,
            -0.15,
            info,
            transform=ax.transAxes,
            ha="center",
            va="top",
            fontsize=12
        )


        ax.axis("off")



    # Remove empty axes
    for ax in axes[n_images:]:
        ax.axis("off")



    plt.suptitle(
        title,
        fontsize=16
    )


    plt.tight_layout()


    # -----------------------------
    # Save figure
    # -----------------------------

    save_path = (
        COMPARISON_DIR
        / f"{filename}.png"
    )

    try:
        plt.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight"
        )
        #plt.show()
    finally:
        plt.close(fig)


    print(
        f"Displayed {n_images} images"
    )

    print(
        f"Saved: {save_path}"
    )

In [80]:
# ==========================================================
# Comparison plotting function with row-dependent columns
# ==========================================================

def plot_fibsem_comparison(
    metadata_filter,
    row_variable,
    column_variable,
    row_order=None,
    column_order=None,
    column_map=None,
    title="Comparison",
    filename="comparison"
):

    selected = metadata[
        metadata_filter
    ].copy()


    if selected.empty:
        print("No matching images found.")
        return


    # ------------------------------------------------------
    # Ordering rows
    # ------------------------------------------------------

    if row_order is None:

        row_values = sorted(
            selected[row_variable].unique()
        )

    else:

        row_values = row_order



    # ------------------------------------------------------
    # Ordering columns
    # ------------------------------------------------------

    if column_order is None:

        column_values = sorted(
            selected[column_variable].unique()
        )

    else:

        column_values = column_order



    # ------------------------------------------------------
    # Determine number of columns
    # ------------------------------------------------------

    if column_map is not None:

        max_columns = max(
            len(values)
            for values in column_map.values()
        )

    else:

        max_columns = len(column_values)



    # ------------------------------------------------------
    # Create figure
    # ------------------------------------------------------

    plt.close("all")

    fig, axes = plt.subplots(

        len(row_values),

        max_columns,

        figsize=(
            5 * max_columns,
            6 * len(row_values)
        ),

        dpi=400

    )


    # Handle axes shape

    if len(row_values) == 1 and max_columns == 1:

        axes = [[axes]]

    elif len(row_values) == 1:

        axes = [axes]

    elif max_columns == 1:

        axes = [[ax] for ax in axes]



    # ------------------------------------------------------
    # Fill figure
    # ------------------------------------------------------

    for row_index, row_value in enumerate(row_values):


        # Select columns for this row
        if column_map is not None:

            current_columns = column_map[row_value]

        else:

            current_columns = column_values



        for col_index, col_value in enumerate(current_columns):


            ax = axes[row_index][col_index]


            subset = selected[

                (selected[row_variable] == row_value)

                &

                (selected[column_variable] == col_value)

            ]



            if subset.empty:

                ax.axis("off")
                continue



            meta = subset.iloc[0]


            image_id = meta["image_id"]



            # --------------------------------------------------
            # Find image
            # --------------------------------------------------

            image_path = None


            for p in RAW_DIR.glob("*.tif"):

                if get_image_id(p) == image_id:

                    image_path = p
                    break



            if image_path is None:

                ax.axis("off")
                continue



            image = tifffile.imread(
                image_path
            )



            ax.imshow(

                image,

                cmap="gray",

                interpolation="nearest",

                aspect="equal"

            )



            # --------------------------------------------------
            # Column heading
            # --------------------------------------------------

            if row_index == 0 or column_map is not None:

                ax.set_title(

                    f"{column_variable}\n{col_value}",

                    fontsize=14,

                    fontweight="bold"

                )



            # --------------------------------------------------
            # Row heading
            # --------------------------------------------------

            if col_index == 0:

                ax.set_ylabel(

                    f"{row_variable}\n{row_value}",

                    fontsize=14,

                    fontweight="bold",

                    rotation=0,

                    labelpad=55,

                    va="center"

                )



            # --------------------------------------------------
            # Metadata text
            # --------------------------------------------------

            info = (

                f"ID: {image_id}\n" 

                +

                f"Detector: {meta.get('detector','N/A')}\n"

                + (
                    f"Mode: {meta.get('mode','N/A')}\n"
                    if pd.notna(meta.get('mode'))
                    and str(meta.get('mode')).strip().upper() != "NA"
                    else ""
                )
                +

                f"Current: {meta.get('current_nA','N/A')} nA\n"

                +

                f"Voltage: {meta.get('voltage_kV','N/A')} kV\n"

                +
                f"Magnification: {meta.get('magnification','N/A')}"
            )



            ax.text(

                0.5,

                -0.10,

                info,

                transform=ax.transAxes,

                ha="center",

                va="top",

                fontsize=11

            )



            ax.set_xticks([])
            ax.set_yticks([])



    # Hide unused axes

    for row in axes:

        for ax in row:

            if not ax.has_data():

                ax.axis("off")



    # ------------------------------------------------------
    # Figure title
    # ------------------------------------------------------

    plt.suptitle(

        title,

        fontsize=18,

        fontweight="bold"

    )


    plt.tight_layout()



    # ------------------------------------------------------
    # Save
    # ------------------------------------------------------

    save_path = (

        COMPARISON_DIR

        / f"{filename}.png"

    )


    try:

        plt.savefig(

            save_path,

            dpi=400,

            bbox_inches="tight"

        )

    finally:

        plt.close(fig)



    print(
        f"Saved to:\n{save_path}"
    )

In [81]:
plot_fibsem_images(
    metadata["parameter_set"] == 1,
    title="Parameter Set 1",
    filename="Parameter_Set_1_current_comparison"
)

Displayed 10 images
Saved: c:\PhD_programs\Opetemisation_data\code\comparison_figures\Parameter_Set_1_current_comparison.png


In [82]:
plot_fibsem_images(
    (metadata["parameter_set"] == 2) &
    (metadata["current_nA"] == 0.1),
    title="Parameter Set 2 | 0.1 nA",
    filename="Parameter_Set_2_Current_0.1nA"
)

Displayed 5 images
Saved: c:\PhD_programs\Opetemisation_data\code\comparison_figures\Parameter_Set_2_Current_0.1nA.png


In [83]:
plot_fibsem_images(
    (metadata["parameter_set"] == 2) &
    (metadata["current_nA"] == 0.0031),
    title="Parameter Set 2 | 0.0031 nA",
    filename="Parameter_Set_2_Current_0.0031nA"
)

Displayed 6 images
Saved: c:\PhD_programs\Opetemisation_data\code\comparison_figures\Parameter_Set_2_Current_0.0031nA.png


In [84]:
plot_fibsem_images(
    (metadata["parameter_set"] == 2),
    title="Parameter Set 2 | all currents",
    filename="Parameter_Set_2_Current_all"
)

Displayed 11 images
Saved: c:\PhD_programs\Opetemisation_data\code\comparison_figures\Parameter_Set_2_Current_all.png


In [85]:
plot_fibsem_comparison(

    metadata_filter=
        metadata["parameter_set"] == 2,

    row_variable="current_nA",

    column_variable="voltage_kV",

    row_order=[0.1, 0.0031],

    column_order=[5, 4, 3, 2, 1],

    title="Parameter Set 2\nVoltage & Current Comparison",

    filename="Parameter_Set_2_Voltage_Current_Comparison"

)

Saved to:
c:\PhD_programs\Opetemisation_data\code\comparison_figures\Parameter_Set_2_Voltage_Current_Comparison.png


In [86]:
plot_fibsem_images(
    (metadata["parameter_set"] == 3),
    title="Parameter Set 3 | Tube Voltage Comparison",
    filename="Parameter_Set_3_Tube_Voltage_Comparison",
    suction_voltage=True
)

Displayed 12 images
Saved: c:\PhD_programs\Opetemisation_data\code\comparison_figures\Parameter_Set_3_Tube_Voltage_Comparison.png


In [87]:
plot_fibsem_comparison(

    metadata_filter=
        metadata["parameter_set"] == 4,

    row_variable="magnification",

    column_variable="voltage_kV",

    row_order=["50000x", "100000x"],

    column_order=[5, 10, 15, 20],

    title="Parameter Set 4\nVoltage & Magnification Comparison",

    filename="Parameter_Set_4_Voltage_Magnification_Comparison"

)

Saved to:
c:\PhD_programs\Opetemisation_data\code\comparison_figures\Parameter_Set_4_Voltage_Magnification_Comparison.png


In [88]:
plot_fibsem_images(
    (metadata["parameter_set"] == 5),
    title="Parameter Set 5\nCurrent Comparison",
    filename="Parameter_Set_5_Current_Comparison",
    signal_out=True,
)

Displayed 5 images
Saved: c:\PhD_programs\Opetemisation_data\code\comparison_figures\Parameter_Set_5_Current_Comparison.png


In [89]:
voltage_current_map = {

    30: [0.04, 0.0011, 0.0077, 0.024],

    15: [0.022, 0.00096, 0.0038, 0.011]

}


plot_fibsem_comparison(

    metadata_filter=(
        metadata["parameter_set"] == 6
    ),

    row_variable="voltage_kV",

    column_variable="current_nA",

    row_order=[
        30,
        15
    ],

    column_map=voltage_current_map,

    title="Parameter Set 6\nVoltage & Current Comparison",

    filename="Parameter_Set_6_Voltage_Current_Comparison"

)

Saved to:
c:\PhD_programs\Opetemisation_data\code\comparison_figures\Parameter_Set_6_Voltage_Current_Comparison.png


In [90]:
plot_fibsem_images(
    (metadata["parameter_set"] == 7),
    title="Parameter Set 7\nSame settings but different areas",
    filename="Parameter_Set_7_area_Comparison",
    signal_out=True,
)

Displayed 10 images
Saved: c:\PhD_programs\Opetemisation_data\code\comparison_figures\Parameter_Set_7_area_Comparison.png


In [ ]:
plot_fibsem_images(
    (metadata["parameter_set"] == 8) & (metadata["voltage_kV"] == 5),
    title="Parameter Set 8\n ETD Current comparison",
    filename="Parameter_Set_8_ETD_Current_Comparison",
    signal_out=True,
)

Displayed 7 images
Saved: c:\PhD_programs\Opetemisation_data\code\comparison_figures\Parameter_Set_8_current_Comparison.png


In [96]:
plot_fibsem_images(
    (metadata["parameter_set"] == 8) & (metadata["current_nA"] == 0.8) & (metadata["image_id"].between(70, 73)),
    title="Parameter Set 8\n ETD Voltage comparison",
    filename="Parameter_Set_8_ETD_Voltage_Comparison",
    signal_out=True,
)

Displayed 4 images
Saved: c:\PhD_programs\Opetemisation_data\code\comparison_figures\Parameter_Set_8_ETD_Voltage_Comparison.png


In [97]:
plot_fibsem_images(
    (metadata["parameter_set"] == 8) & (metadata["voltage_kV"] == 30) & (metadata["image_id"].between(73, 77)),
    title="Parameter Set 8\n ETD Current at 30 kV comparison",
    filename="Parameter_Set_8_ETD_Current_30kV_Comparison",
    signal_out=True,
)

Displayed 5 images
Saved: c:\PhD_programs\Opetemisation_data\code\comparison_figures\Parameter_Set_8_ETD_Current_30kV_Comparison.png


In [98]:
plot_fibsem_images(
    (metadata["parameter_set"] == 8) & (metadata["voltage_kV"] == 10) & (metadata["image_id"].between(78, 80)),
    title="Parameter Set 8\n ETD Current at 10 kV comparison",
    filename="Parameter_Set_8_ETD_Current_10kV_Comparison",
    signal_out=True,
)

Displayed 3 images
Saved: c:\PhD_programs\Opetemisation_data\code\comparison_figures\Parameter_Set_8_ETD_Current_10kV_Comparison.png


In [99]:
plot_fibsem_images(
    (metadata["parameter_set"] == 8) & (metadata["voltage_kV"] == 15) & (metadata["image_id"].between(81, 82)),
    title="Parameter Set 8\n ETD Current at 15 kV comparison",
    filename="Parameter_Set_8_ETD_Current_15kV_Comparison",
    signal_out=True,
)

Displayed 2 images
Saved: c:\PhD_programs\Opetemisation_data\code\comparison_figures\Parameter_Set_8_ETD_Current_15kV_Comparison.png


In [100]:
plot_fibsem_images(
    (metadata["parameter_set"] == 8) & (metadata["voltage_kV"] == 20) & (metadata["image_id"].between(83, 85)),
    title="Parameter Set 8\n ETD Current at 20 kV comparison",
    filename="Parameter_Set_8_ETD_Current_20kV_Comparison",
    signal_out=True,
)

Displayed 3 images
Saved: c:\PhD_programs\Opetemisation_data\code\comparison_figures\Parameter_Set_8_ETD_Current_20kV_Comparison.png


In [ ]:
voltage_current_map = {

    5: [0,8, 1.6, 0.8, 0.4, 0.2, 3.2, 0.8],

    10: [0.8, 1.6, 1.6, 0.4],

    15: [0.8, 0.4, 1.6],

    20: [0.8, 1.6, 0.8, 0.4],

    30: [0.8, 1.6, 1.6, 3.2, 0.4]

}

plot_fibsem_comparison(

    metadata_filter=
        metadata["parameter_set"] == 8,

    row_variable="voltage_kV",

    column_variable="current_nA",

    row_order=[5, 10, 15, 20, 30],

    column_map=voltage_current_map,

    title="Parameter Set 8\nVoltage & Current Comparison",

    filename="Parameter_Set_8_Voltage_Current_Comparison"

)

Saved to:
c:\PhD_programs\Opetemisation_data\code\comparison_figures\Parameter_Set_8_Voltage_Current_Comparison.png
